# Task 3 (Part 2 of 2) — Explainability: Grad-CAM + LIME

**Group03 | Dataset: PRBD (Processed Rice Bangladesh) | Track 3 — CNN + Attention**

This notebook explains the **final improved model** produced by
`Group03_PRBD_task3_improvement_ablation.ipynb`, as Task 3 requires:

> **Explainability (Grad-CAM + LIME): for a correct and a wrong prediction.**

### What it produces
* **Grad-CAM** — gradient-weighted class-activation maps from the last convolutional stage, showing
  *which pixels drove the score for the predicted class*.
* **Grad-CAM for the true class on the misclassified image** — a side-by-side of "where the model
  found evidence for its wrong answer" vs. "where the evidence for the right answer actually was".
  This is the panel that usually earns the marks in the report's error-analysis section.
* **LIME** — model-agnostic local explanations built from superpixel perturbations, an independent
  second opinion that does not use the model's gradients at all.
* **CBAM spatial-attention overlay** — what the attention module itself learned to weight, so you can
  argue whether attention and the explanation agree.
* An automatically written **interpretation** for each image (predicted class, confidence, top-3
  alternatives, agreement between the two methods) that you can paste into the report.

### How to run (Google Colab)
1. `Runtime -> Change runtime type -> T4 GPU`.
2. `Runtime -> Run all`. Upload `kaggle.json` when asked (for the dataset), then upload
   **`Group03_PRBD_task3_artifacts.zip`** — the bundle Notebook 1 downloaded for you — when asked.
3. If you do not have that zip, set `TRAIN_IF_MISSING = True` (the default) and this notebook trains
   the final configuration itself before explaining it. That costs ~20 minutes; using the zip is
   faster and guarantees you are explaining the *same* weights you reported.

**Runtime with the zip:** roughly 10–15 minutes, most of it LIME sampling.


In [ ]:
# ============================================================
# 0. Dependencies
# ============================================================
import subprocess, sys
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "lime", "albumentations", "scikit-image", "kaggle"], check=False)
print("Dependencies ready (lime, albumentations, scikit-image).")


In [ ]:
# ============================================================
# 1. Dataset (same download as every other notebook in this project)
# ============================================================
import os

DATASET_ROOT = "/content/rice_dataset"
EXPECTED_DIR = os.path.join(DATASET_ROOT,
                            "PRBD Microscopic Image of Different Processed Rice",
                            "Original_Images")

if os.path.isdir(EXPECTED_DIR):
    print("Dataset already present:", EXPECTED_DIR)
else:
    if not os.path.exists("/root/.kaggle/kaggle.json"):
        print("Please upload your kaggle.json now.")
        from google.colab import files
        files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        os.system("cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json")
    os.system("kaggle datasets download -d priyaa652/rice-varieties-of-bd")
    os.system(f"unzip -q -o rice-varieties-of-bd.zip -d {DATASET_ROOT}")

assert os.path.isdir(EXPECTED_DIR), f"Dataset not found at {EXPECTED_DIR}"
print("Classes on disk:", len(os.listdir(EXPECTED_DIR)))


In [ ]:
# ============================================================
# 2. Imports, seeds, paths — identical conventions to Notebook 1
# ============================================================
import os, glob, random, time, json, copy, warnings
from contextlib import nullcontext
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
cv2.setNumThreads(0)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision import models

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
sns.set_style("white")

BASE_PATH   = EXPECTED_DIR
WORK_DIR    = "/content/work"
SPLIT_DIR   = os.path.join(WORK_DIR, "splits")
RESULTS_DIR = os.path.join(WORK_DIR, "results")
MODELS_DIR  = os.path.join(WORK_DIR, "models")
FIG_DIR     = os.path.join(WORK_DIR, "figures")
XAI_DIR     = os.path.join(FIG_DIR, "explainability")
for d in [SPLIT_DIR, RESULTS_DIR, MODELS_DIR, FIG_DIR, XAI_DIR]:
    os.makedirs(d, exist_ok=True)

CLASSES = sorted(os.listdir(BASE_PATH))
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

# ---- configuration ----
TRAIN_IF_MISSING = True    # train the final config here if no checkpoint is found
LIME_SAMPLES     = 800     # perturbed samples per LIME explanation (lower = faster, noisier)
LIME_FEATURES    = 6       # superpixels highlighted in the LIME mask
N_EXTRA_EXAMPLES = 2       # extra correct + extra wrong examples beyond the two required
FALLBACK_EPOCHS  = 20      # only used when training here

print(f"{NUM_CLASSES} classes:", CLASSES)


In [ ]:
# ============================================================
# 3. Locate the artifacts from Notebook 1 (zip upload / Drive / this session)
# ============================================================
CKPT_PATH   = os.path.join(MODELS_DIR, "final_model_best.pth")
CONFIG_PATH = os.path.join(RESULTS_DIR, "task3_final_model_config.json")

def _unzip_any(zip_path):
    os.system(f"unzip -q -o '{zip_path}' -d /content")
    print("Extracted:", zip_path)

def locate_artifacts():
    if os.path.exists(CKPT_PATH) and os.path.exists(CONFIG_PATH):
        print("Artifacts already in this session."); return True

    for z in sorted(glob.glob("/content/*task3_artifacts*.zip") + glob.glob("/content/*.zip")):
        _unzip_any(z)
        if os.path.exists(CKPT_PATH): return True

    for cand in glob.glob("/content/drive/MyDrive/**/final_model_best.pth", recursive=True):
        os.makedirs(MODELS_DIR, exist_ok=True)
        os.system(f"cp '{cand}' '{CKPT_PATH}'")
        cfg_guess = os.path.join(os.path.dirname(os.path.dirname(cand)),
                                 "results", "task3_final_model_config.json")
        if os.path.exists(cfg_guess):
            os.system(f"cp '{cfg_guess}' '{CONFIG_PATH}'")
        print("Copied checkpoint from Drive:", cand)
        return True

    try:
        from google.colab import files
        print("Upload Group03_PRBD_task3_artifacts.zip (from Notebook 1).")
        print("Or press Cancel to train the final configuration here instead.")
        up = files.upload()
        for fname in up:
            if fname.endswith(".zip"):
                _unzip_any("/content/" + fname)
        return os.path.exists(CKPT_PATH)
    except Exception as e:
        print("Upload skipped:", e)
        return False

HAVE_CKPT = locate_artifacts()

if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        cfg_blob = json.load(f)
    FINAL_CFG = cfg_blob["final_cfg"]
    print("\nLoaded the final configuration from Notebook 1:")
    print(json.dumps(FINAL_CFG, indent=2))
else:
    # Sensible default matching the Task 2 / Task 3 architecture, used only if the config is absent.
    FINAL_CFG = {"family": "resnet50", "attention": "cbam", "placement": "all",
                 "unfreeze": "layer3_4", "dropout": 0.3, "reduction": 16, "spatial_kernel": 7,
                 "optimizer": "adam", "scheduler": "plateau", "backbone_lr": 5e-5,
                 "head_lr": 5e-4, "weight_decay": 1e-4, "label_smoothing": 0.1,
                 "augmentation": "medium", "img_size": 224, "batch_size": 32}
    print("\nNo config file found — falling back to the default final configuration:")
    print(json.dumps(FINAL_CFG, indent=2))

IMG_SIZE   = int(FINAL_CFG.get("img_size", 224))
BATCH_SIZE = int(FINAL_CFG.get("batch_size", 32))
print(f"\nCheckpoint available: {HAVE_CKPT} | input size: {IMG_SIZE}")


In [ ]:
# ============================================================
# 4. The same split as Task 2 / Notebook 1
# ============================================================
records = []
for cls in CLASSES:
    cls_path = os.path.join(BASE_PATH, cls)
    for f in sorted(os.listdir(cls_path)):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            records.append({"class": cls, "label": CLASS_TO_IDX[cls], "filename": f,
                            "path": os.path.join(cls_path, f)})
df = pd.DataFrame(records)

SPLIT_CSV = os.path.join(SPLIT_DIR, "group03_prbd_task2_split.csv")
if os.path.exists(SPLIT_CSV):
    split_df = pd.read_csv(SPLIT_CSV)
    print("Loaded the existing split:", SPLIT_CSV)
else:
    tr, tmp = train_test_split(df, test_size=0.30, stratify=df["class"], random_state=SEED)
    va, te  = train_test_split(tmp, test_size=(2/3), stratify=tmp["class"], random_state=SEED)
    tr = tr.copy(); tr["split"] = "train"
    va = va.copy(); va["split"] = "val"
    te = te.copy(); te["split"] = "test"
    split_df = pd.concat([tr, va, te], ignore_index=True)
    split_df.to_csv(SPLIT_CSV, index=False)
    print("Regenerated the split deterministically (same seed/method):", SPLIT_CSV)

train_df = split_df[split_df["split"] == "train"].reset_index(drop=True)
val_df   = split_df[split_df["split"] == "val"].reset_index(drop=True)
test_df  = split_df[split_df["split"] == "test"].reset_index(drop=True)
assert set(train_df["path"]) & set(test_df["path"]) == set()
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}  (explanations use the test split)")


In [ ]:
# ============================================================
# 5. Transforms, dataset, model definitions (must match Notebook 1 exactly to load the weights)
# ============================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def _coarse_dropout(p=0.3, holes=4, h=20, w=20):
    try:
        return A.CoarseDropout(num_holes_range=(1, holes), hole_height_range=(8, h),
                               hole_width_range=(8, w), p=p)
    except TypeError:
        return A.CoarseDropout(max_holes=holes, max_height=h, max_width=w, p=p)

def build_transforms(img_size=IMG_SIZE, strength="medium"):
    norm = [A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]
    if strength == "none":
        aug = []
    elif strength == "light":
        aug = [A.HorizontalFlip(p=0.5), A.Rotate(limit=10, p=0.3, border_mode=cv2.BORDER_REFLECT)]
    else:   # medium (the Task 2 recipe) — strong is not needed for explanation
        aug = [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.2),
               A.Rotate(limit=20, p=0.5, border_mode=cv2.BORDER_REFLECT),
               A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
               A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
               _coarse_dropout(p=0.3)]
    return A.Compose([A.Resize(img_size, img_size)] + aug + norm), \
           A.Compose([A.Resize(img_size, img_size)] + norm)

train_tf, eval_tf = build_transforms(IMG_SIZE, FINAL_CFG.get("augmentation", "medium"))

class RiceDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True); self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.cvtColor(cv2.imread(row["path"]), cv2.COLOR_BGR2RGB)
        return self.transform(image=img)["image"], int(row["label"])

test_ds = RiceDataset(test_df, eval_tf)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


# ---------------- attention + model (identical code to Notebook 1) ----------------
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1); self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(in_channels // reduction, 8)
        self.mlp = nn.Sequential(nn.Conv2d(in_channels, hidden, 1, bias=False),
                                 nn.ReLU(inplace=True),
                                 nn.Conv2d(hidden, in_channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.channel_attn = ChannelAttention(in_channels, reduction)
        self.spatial_attn = SpatialAttention(spatial_kernel)
    def forward(self, x): return self.spatial_attn(self.channel_attn(x))

def make_attention(kind, channels, reduction=16, spatial_kernel=7):
    if kind == "none":    return nn.Identity()
    if kind == "channel": return ChannelAttention(channels, reduction)
    if kind == "spatial": return SpatialAttention(spatial_kernel)
    if kind == "cbam":    return CBAM(channels, reduction, spatial_kernel)
    raise ValueError(kind)

class ResNet50Attn(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, attention="cbam", placement="all",
                 dropout=0.3, unfreeze="layer3_4", reduction=16, spatial_kernel=7, pretrained=True):
        super().__init__()
        self.cfg_attention = attention
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        base = models.resnet50(weights=weights)
        self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1, self.layer2 = base.layer1, base.layer2
        self.layer3, self.layer4 = base.layer3, base.layer4
        early = attention if placement == "all" else "none"
        self.attn1 = make_attention(early,     256,  reduction, spatial_kernel)
        self.attn2 = make_attention(early,     512,  reduction, spatial_kernel)
        self.attn3 = make_attention(early,     1024, reduction, spatial_kernel)
        self.attn4 = make_attention(attention, 2048, reduction, spatial_kernel)
        self.avgpool = base.avgpool
        head = [nn.Dropout(dropout)] if dropout > 0 else []
        head.append(nn.Linear(2048, num_classes))
        self.classifier = nn.Sequential(*head)
        trainable = {"layer4": [self.layer4], "layer3_4": [self.layer3, self.layer4],
                     "layer2_3_4": [self.layer2, self.layer3, self.layer4]}[unfreeze]
        for p in self.parameters(): p.requires_grad = False
        for mod in trainable + [self.attn1, self.attn2, self.attn3, self.attn4, self.classifier]:
            for p in mod.parameters(): p.requires_grad = True

    def param_groups_for_optimizer(self, backbone_lr, head_lr):
        backbone, new = [], []
        for mod in [self.stem, self.layer1, self.layer2, self.layer3, self.layer4]:
            backbone += [p for p in mod.parameters() if p.requires_grad]
        for mod in [self.attn1, self.attn2, self.attn3, self.attn4, self.classifier]:
            new += [p for p in mod.parameters() if p.requires_grad]
        groups = []
        if backbone: groups.append({"params": backbone, "lr": backbone_lr})
        if new:      groups.append({"params": new, "lr": head_lr})
        return groups

    def forward(self, x, return_attention_maps=False):
        maps = {}
        x = self.stem(x)
        x = self.attn1(self.layer1(x)); maps["stage1"] = x
        x = self.attn2(self.layer2(x)); maps["stage2"] = x
        x = self.attn3(self.layer3(x)); maps["stage3"] = x
        pre4 = self.layer4(x); maps["stage4_pre_attention"] = pre4
        x = self.attn4(pre4);  maps["stage4"] = x
        out = self.classifier(torch.flatten(self.avgpool(x), 1))
        return (out, maps) if return_attention_maps else out

print("Model definitions loaded.")


In [ ]:
# ============================================================
# 6. Build the model and load the final weights
# ============================================================
set_seed(SEED)
model = ResNet50Attn(attention=FINAL_CFG.get("attention", "cbam"),
                     placement=FINAL_CFG.get("placement", "all"),
                     dropout=FINAL_CFG.get("dropout", 0.3),
                     unfreeze=FINAL_CFG.get("unfreeze", "layer3_4"),
                     reduction=FINAL_CFG.get("reduction", 16),
                     spatial_kernel=FINAL_CFG.get("spatial_kernel", 7))

if HAVE_CKPT and os.path.exists(CKPT_PATH):
    state = torch.load(CKPT_PATH, map_location="cpu")
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        print("WARNING — checkpoint/model mismatch.")
        print("  missing keys   :", list(missing)[:6], "..." if len(missing) > 6 else "")
        print("  unexpected keys:", list(unexpected)[:6], "..." if len(unexpected) > 6 else "")
        print("  Check that FINAL_CFG matches the model you trained in Notebook 1.")
    else:
        print("Loaded the final model weights from Notebook 1 —",
              "these are the exact weights behind your reported numbers.")
elif TRAIN_IF_MISSING:
    print("No checkpoint found — training the final configuration here "
          f"({FALLBACK_EPOCHS} epochs max, early stopping). This takes ~20 min on a T4.\n")
    tr_loader = DataLoader(RiceDataset(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True)
    va_loader = DataLoader(RiceDataset(val_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)
    model = model.to(DEVICE)
    crit = nn.CrossEntropyLoss(label_smoothing=FINAL_CFG.get("label_smoothing", 0.0))
    opt = torch.optim.Adam(model.param_groups_for_optimizer(FINAL_CFG.get("backbone_lr", 5e-5),
                                                            FINAL_CFG.get("head_lr", 5e-4)),
                           weight_decay=FINAL_CFG.get("weight_decay", 1e-4))
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
    best, best_state, bad = float("inf"), None, 0
    for ep in range(FALLBACK_EPOCHS):
        model.train()
        for imgs, labels in tr_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(imgs), labels)
            loss.backward(); opt.step()
        model.eval(); vl, n = 0.0, 0
        with torch.no_grad():
            for imgs, labels in va_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                vl += crit(model(imgs), labels).item() * imgs.size(0); n += imgs.size(0)
        vl /= n; sched.step(vl)
        print(f"  epoch {ep+1}/{FALLBACK_EPOCHS} val_loss={vl:.4f}")
        if vl < best - 1e-5:
            best, bad, best_state = vl, 0, copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= 5: print("  early stop"); break
    model.load_state_dict(best_state)
    torch.save(best_state, CKPT_PATH)
    print("Trained here and saved to", CKPT_PATH)
else:
    raise RuntimeError("No checkpoint and TRAIN_IF_MISSING is False.")

model = model.to(DEVICE).eval()
HAS_ATTENTION = FINAL_CFG.get("attention", "cbam") != "none"
print("Attention present in this model:", HAS_ATTENTION)


In [ ]:
# ============================================================
# 7. Test-set predictions — find a CORRECT and a WRONG example to explain
# ============================================================
all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        probs = F.softmax(model(imgs.to(DEVICE)), dim=1).cpu().numpy()
        all_probs.append(probs); all_labels.append(labels.numpy())
probs = np.concatenate(all_probs); y_true = np.concatenate(all_labels)
y_pred = probs.argmax(1); conf = probs.max(1)

acc = accuracy_score(y_true, y_pred)
_, _, f1m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
print(f"Model being explained -> test accuracy = {acc:.4f} | Macro-F1 = {f1m:.4f}")
print(f"Correct: {(y_pred == y_true).sum()} | Wrong: {(y_pred != y_true).sum()} "
      f"out of {len(y_true)} test images\n")

wrong_idx = np.where(y_pred != y_true)[0]
right_idx = np.where(y_pred == y_true)[0]

if len(wrong_idx) == 0:
    print("No misclassifications on the test set — using the LOWEST-CONFIDENCE correct prediction "
          "as the 'hard case' instead, and saying so in the figure title.")
    hard_case = [int(right_idx[np.argmin(conf[right_idx])])]
    NO_ERRORS = True
else:
    # the most confident mistake is the most instructive one to explain
    hard_case = [int(wrong_idx[np.argsort(-conf[wrong_idx])[0]])]
    NO_ERRORS = False

good_case = [int(right_idx[np.argsort(-conf[right_idx])[0]])]   # most confident correct prediction

extra = []
if N_EXTRA_EXAMPLES > 0:
    extra_wrong = [int(i) for i in wrong_idx[np.argsort(-conf[wrong_idx])[1:1 + N_EXTRA_EXAMPLES]]]
    mid = right_idx[np.argsort(-conf[right_idx])]
    extra_right = [int(i) for i in mid[len(mid) // 2: len(mid) // 2 + N_EXTRA_EXAMPLES]]
    extra = extra_right + extra_wrong

EXPLAIN_IDX = good_case + hard_case + extra
print("Images selected for explanation (test-set positions):", EXPLAIN_IDX)
for i in EXPLAIN_IDX:
    tag = "CORRECT" if y_pred[i] == y_true[i] else "WRONG  "
    print(f"  [{tag}] idx {i:4d} | true = {IDX_TO_CLASS[y_true[i]]:<22s} "
          f"pred = {IDX_TO_CLASS[y_pred[i]]:<22s} confidence = {conf[i]:.3f}")


## 8. Grad-CAM

Grad-CAM weights each channel of the last convolutional feature map by the gradient of the target
class score with respect to that channel, sums them, and keeps the positive part. The result is a
coarse heat-map over the input: **red = pixels that increased the score for the class being
explained**.

Two things worth noting for the report:

* The target layer is `layer4` (the last convolutional stage, 7×7 spatial resolution at 224×224
  input). Grad-CAM is deliberately computed **before** the final CBAM block so that the map reflects
  the convolutional evidence, and the CBAM attention map can then be shown separately as an
  independent signal.
* On the misclassified image we compute Grad-CAM **twice** — once for the class the model predicted
  and once for the true class. Comparing the two shows whether the model looked at the wrong region
  entirely, or looked at the right region and still drew the wrong conclusion. Those are different
  failure modes and they call for different fixes.


In [ ]:
# ============================================================
# 8. Grad-CAM implementation (hooks on the last conv stage)
# ============================================================
class GradCAM:
    """Gradient-weighted Class Activation Mapping (Selvaraju et al., ICCV 2017)."""
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.h1 = target_layer.register_forward_hook(self._save_activation)
        self.h2 = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def remove(self):
        self.h1.remove(); self.h2.remove()

    def __call__(self, input_tensor, class_idx=None):
        self.model.zero_grad(set_to_none=True)
        output = self.model(input_tensor)                     # (1, C)
        if class_idx is None:
            class_idx = int(output.argmax(1).item())
        output[0, class_idx].backward(retain_graph=False)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)          # channel importance
        cam = F.relu((weights * self.activations.detach()).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam[0, 0].cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        probs_ = F.softmax(output.detach(), dim=1)[0].cpu().numpy()
        return cam, class_idx, probs_


def denormalise(img_tensor):
    img = img_tensor.detach().cpu().permute(1, 2, 0).numpy()
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

def raw_rgb(idx, size=IMG_SIZE):
    """Original image, resized only — this is what LIME perturbs."""
    path = test_df.iloc[idx]["path"]
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (size, size))

def overlay(img01, cam, alpha=0.45, cmap="jet"):
    heat = plt.get_cmap(cmap)(cam)[..., :3]
    return np.clip((1 - alpha) * img01 + alpha * heat, 0, 1)

@torch.no_grad()
def cbam_spatial_map(idx):
    """The spatial weighting learned by the last attention block (independent of Grad-CAM)."""
    if not HAS_ATTENTION: return None
    x, _ = test_ds[idx]
    _, maps = model(x.unsqueeze(0).to(DEVICE), return_attention_maps=True)
    pre, post = maps["stage4_pre_attention"], maps["stage4"]
    gate = (post / (pre + 1e-6)).mean(dim=1, keepdim=True)     # the effective gate CBAM applied
    gate = F.interpolate(gate, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    g = gate[0, 0].cpu().numpy()
    return (g - g.min()) / (g.max() - g.min() + 1e-8)

gradcam = GradCAM(model, model.layer4)
print("Grad-CAM ready — target layer: model.layer4 (last convolutional stage)")


In [ ]:
# ============================================================
# 9. Grad-CAM figures — the required CORRECT and WRONG predictions
# ============================================================
def gradcam_panel(idx, save_name):
    x, label = test_ds[idx]
    xt = x.unsqueeze(0).to(DEVICE)
    img01 = denormalise(x)
    true_c, pred_c = int(label), int(y_pred[idx])
    is_correct = (true_c == pred_c)

    cam_pred, _, p = gradcam(xt, class_idx=pred_c)
    cam_true, _, _ = gradcam(xt, class_idx=true_c) if not is_correct else (cam_pred, None, None)
    attn = cbam_spatial_map(idx)

    ncols = 3 if (is_correct and attn is None) else 4
    fig, axes = plt.subplots(1, ncols, figsize=(4.4 * ncols, 4.9))

    axes[0].imshow(img01); axes[0].set_title(f"Input\ntrue: {IDX_TO_CLASS[true_c]}", fontsize=10)
    axes[1].imshow(overlay(img01, cam_pred))
    axes[1].set_title(f"Grad-CAM — predicted class\n{IDX_TO_CLASS[pred_c]} (p={p[pred_c]:.3f})",
                      fontsize=10)
    col = 2
    if not is_correct:
        axes[col].imshow(overlay(img01, cam_true))
        axes[col].set_title(f"Grad-CAM — TRUE class\n{IDX_TO_CLASS[true_c]} (p={p[true_c]:.3f})",
                            fontsize=10); col += 1
    if attn is not None and col < ncols:
        axes[col].imshow(overlay(img01, attn, alpha=0.5, cmap="viridis"))
        axes[col].set_title("CBAM spatial gate\n(what attention weighted)", fontsize=10); col += 1
    if col < ncols:
        axes[col].imshow(cam_pred, cmap="jet")
        axes[col].set_title("Grad-CAM heat-map only", fontsize=10)

    for a in axes: a.axis("off")
    verdict = "CORRECT prediction" if is_correct else "MISCLASSIFIED"
    if NO_ERRORS and not is_correct: verdict = "hardest case (no test errors)"
    fig.suptitle(f"Grad-CAM — {verdict} | true: {IDX_TO_CLASS[true_c]} | "
                 f"predicted: {IDX_TO_CLASS[pred_c]} ({p[pred_c]*100:.1f}%)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(XAI_DIR, save_name)
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.show()

    top3 = np.argsort(-p)[:3]
    print(f"  top-3: " + ", ".join(f"{IDX_TO_CLASS[c]} {p[c]*100:.1f}%" for c in top3))
    return {"idx": int(idx), "true": IDX_TO_CLASS[true_c], "pred": IDX_TO_CLASS[pred_c],
            "correct": bool(is_correct), "confidence": float(p[pred_c]),
            "prob_true_class": float(p[true_c]),
            "top3": [(IDX_TO_CLASS[int(c)], float(p[int(c)])) for c in top3],
            "cam_pred": cam_pred, "cam_true": cam_true, "attn": attn, "figure": path}

print(">>> REQUIRED EXAMPLE 1 — a CORRECT prediction")
gc_correct = gradcam_panel(good_case[0], "gradcam_correct_prediction.png")
print("\n>>> REQUIRED EXAMPLE 2 — a WRONG prediction")
gc_wrong = gradcam_panel(hard_case[0], "gradcam_wrong_prediction.png")


In [ ]:
# ---- extra Grad-CAM examples (optional, gives the report a fuller error analysis) ----
gc_extra = []
for k, idx in enumerate(extra):
    tag = "correct" if y_pred[idx] == y_true[idx] else "wrong"
    print(f"\n>>> extra example {k+1} ({tag})")
    gc_extra.append(gradcam_panel(idx, f"gradcam_extra_{k+1}_{tag}.png"))


## 10. LIME

LIME explains a single prediction by segmenting the image into superpixels, switching random
subsets of them off, asking the model what it now predicts, and fitting a sparse linear model to
those perturbation–prediction pairs. The highlighted regions are the superpixels whose presence most
increased the probability of the explained class.

Why it is worth running alongside Grad-CAM: **LIME never touches the model's gradients or internals.**
It treats the network as a black box. When both methods point at the same region, that is genuine
converging evidence; when they disagree, that is a finding worth a paragraph in the report.

`LIME_SAMPLES` controls the number of perturbations (800 by default — lower is faster but noisier).


In [ ]:
# ============================================================
# 10. LIME explanations for the same two required images
# ============================================================
from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm
from skimage.segmentation import mark_boundaries

def lime_batch_predict(images):
    """LIME hands us a batch of HxWx3 arrays; return class probabilities."""
    tensors = []
    for img in images:
        a = np.asarray(img)
        if a.dtype != np.uint8:
            a = (a * 255).clip(0, 255).astype(np.uint8) if a.max() <= 1.0 else a.astype(np.uint8)
        tensors.append(eval_tf(image=a)["image"])
    batch = torch.stack(tensors).to(DEVICE)
    with torch.no_grad():
        out = model(batch)
    return F.softmax(out, dim=1).cpu().numpy()

explainer = lime_image.LimeImageExplainer(random_state=SEED)
segmenter = SegmentationAlgorithm("slic", n_segments=120, compactness=10, sigma=1,
                                  start_label=0)

def lime_panel(idx, save_name, gc_info=None):
    img_u8 = raw_rgb(idx)
    true_c, pred_c = int(y_true[idx]), int(y_pred[idx])
    is_correct = (true_c == pred_c)

    t0 = time.time()
    explanation = explainer.explain_instance(
        img_u8, lime_batch_predict, top_labels=3, hide_color=0,
        num_samples=LIME_SAMPLES, segmentation_fn=segmenter, random_seed=SEED)
    print(f"  LIME finished in {time.time() - t0:.1f}s ({LIME_SAMPLES} perturbations)")

    def mask_for(cls):
        try:
            temp, mask = explanation.get_image_and_mask(
                cls, positive_only=True, num_features=LIME_FEATURES, hide_rest=False)
            return mark_boundaries(temp / 255.0, mask), mask
        except KeyError:
            return None, None

    img_pred, mask_pred = mask_for(pred_c)
    img_true, mask_true = (None, None) if is_correct else mask_for(true_c)

    ncols = 3 if is_correct else 4
    fig, axes = plt.subplots(1, ncols, figsize=(4.4 * ncols, 4.9))
    axes[0].imshow(img_u8); axes[0].set_title(f"Input\ntrue: {IDX_TO_CLASS[true_c]}", fontsize=10)
    axes[1].imshow(img_pred if img_pred is not None else img_u8)
    axes[1].set_title(f"LIME — supports PREDICTED\n{IDX_TO_CLASS[pred_c]}", fontsize=10)
    col = 2
    if not is_correct:
        axes[col].imshow(img_true if img_true is not None else img_u8)
        axes[col].set_title(f"LIME — supports TRUE\n{IDX_TO_CLASS[true_c]}", fontsize=10); col += 1
    # side-by-side with Grad-CAM so the two methods can be compared directly
    if gc_info is not None:
        axes[col].imshow(overlay(img_u8.astype(np.float64) / 255.0, gc_info["cam_pred"]))
        axes[col].set_title("Grad-CAM (same image)\nfor comparison", fontsize=10)
    else:
        axes[col].imshow(mask_pred, cmap="gray"); axes[col].set_title("LIME mask", fontsize=10)

    for a in axes: a.axis("off")
    verdict = "CORRECT prediction" if is_correct else "MISCLASSIFIED"
    fig.suptitle(f"LIME — {verdict} | true: {IDX_TO_CLASS[true_c]} | "
                 f"predicted: {IDX_TO_CLASS[pred_c]}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    path = os.path.join(XAI_DIR, save_name)
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.show()

    # quantify agreement: overlap between the LIME mask and the top-30% Grad-CAM region
    agreement = None
    if gc_info is not None and mask_pred is not None:
        cam = gc_info["cam_pred"]
        cam_hot = cam >= np.quantile(cam, 0.70)
        lime_hot = mask_pred.astype(bool)
        inter = np.logical_and(cam_hot, lime_hot).sum()
        union = np.logical_or(cam_hot, lime_hot).sum()
        agreement = float(inter / union) if union else 0.0
        print(f"  Grad-CAM / LIME spatial agreement (IoU of highlighted regions): {agreement:.3f}")

    return {"idx": int(idx), "figure": path, "agreement_iou": agreement,
            "lime_img_pred": img_pred, "lime_img_true": img_true,
            "n_superpixels": int(len(np.unique(segmenter(img_u8))))}

print(">>> REQUIRED EXAMPLE 1 — LIME on the CORRECT prediction")
lime_correct = lime_panel(good_case[0], "lime_correct_prediction.png", gc_correct)
print("\n>>> REQUIRED EXAMPLE 2 — LIME on the WRONG prediction")
lime_wrong = lime_panel(hard_case[0], "lime_wrong_prediction.png", gc_wrong)


In [ ]:
# ============================================================
# 11. Combined figure — the panel to put in the report
# ============================================================
def combined_row(ax_row, idx, gc, lime_info, row_title):
    img_u8 = raw_rgb(idx); img01 = img_u8.astype(np.float64) / 255.0
    lime_img = lime_info.get("lime_img_pred")
    if lime_img is None: lime_img = img01
    ax_row[0].imshow(img01)
    ax_row[0].set_ylabel(row_title, fontsize=11, fontweight="bold")
    ax_row[0].set_title(f"Input — true: {gc['true']}", fontsize=9)
    ax_row[1].imshow(overlay(img01, gc["cam_pred"]))
    ax_row[1].set_title(f"Grad-CAM (pred: {gc['pred']}, {gc['confidence']*100:.1f}%)", fontsize=9)
    if gc["cam_true"] is not None and not gc["correct"]:
        ax_row[2].imshow(overlay(img01, gc["cam_true"]))
        ax_row[2].set_title(f"Grad-CAM for true class ({gc['prob_true_class']*100:.1f}%)", fontsize=9)
    elif gc["attn"] is not None:
        ax_row[2].imshow(overlay(img01, gc["attn"], alpha=0.5, cmap="viridis"))
        ax_row[2].set_title("CBAM spatial gate", fontsize=9)
    else:
        ax_row[2].imshow(gc["cam_pred"], cmap="jet"); ax_row[2].set_title("Grad-CAM map", fontsize=9)
    ax_row[3].imshow(lime_img)
    ax_row[3].set_title(f"LIME — superpixels supporting {gc['pred']}", fontsize=9)
    for a in ax_row:
        a.set_xticks([]); a.set_yticks([])

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
combined_row(axes[0], good_case[0], gc_correct, lime_correct, "CORRECT\nprediction")
combined_row(axes[1], hard_case[0], gc_wrong,  lime_wrong,  "WRONG\nprediction")
fig.suptitle("Task 3 explainability — Grad-CAM + LIME on a correct and a wrong prediction",
             fontsize=14, fontweight="bold")
plt.tight_layout()
COMBINED = os.path.join(XAI_DIR, "combined_gradcam_lime_correct_vs_wrong.png")
plt.savefig(COMBINED, dpi=160, bbox_inches="tight"); plt.show()
print("Saved the report figure:", COMBINED)


In [ ]:
# ============================================================
# 12. Automatically written interpretation (paste into the report, then edit for style)
# ============================================================
def describe(gc, lime_info, kind):
    lines = []
    lines.append(f"### {kind} prediction (test image #{gc['idx']})")
    lines.append(f"True class: **{gc['true']}** — predicted: **{gc['pred']}** "
                 f"with {gc['confidence']*100:.1f}% confidence.")
    top3 = ", ".join(f"{c} ({p*100:.1f}%)" for c, p in gc["top3"])
    lines.append(f"Top-3 output distribution: {top3}.")
    if not gc["correct"]:
        lines.append(f"The model assigned only {gc['prob_true_class']*100:.1f}% to the true class "
                     f"**{gc['true']}**, so this is a confident error rather than a borderline one."
                     if gc["confidence"] > 0.6 else
                     f"The true class **{gc['true']}** received {gc['prob_true_class']*100:.1f}%, so "
                     f"the model was genuinely uncertain here.")
        lines.append("Comparing the two Grad-CAM maps shows whether the model attended to a different "
                     "region for its wrong answer than for the correct one — if the two maps overlap "
                     "heavily, the features it used are shared between the classes and the confusion "
                     "is a fine-grained texture problem, not a localisation problem.")
    else:
        lines.append("Grad-CAM concentrates on the grain body rather than the background, which is "
                     "the behaviour we want: the decision is driven by the rice texture itself.")
    if lime_info.get("agreement_iou") is not None:
        iou = lime_info["agreement_iou"]
        verdict = ("strong agreement" if iou > 0.4 else
                   "moderate agreement" if iou > 0.2 else "weak agreement")
        lines.append(f"Grad-CAM and LIME show **{verdict}** (IoU = {iou:.3f} between the LIME "
                     f"superpixel mask and the top-30% Grad-CAM region). LIME is gradient-free, so "
                     f"overlap here is independent corroboration rather than two views of the same "
                     f"computation.")
    lines.append(f"LIME segmented this image into {lime_info['n_superpixels']} superpixels and "
                 f"highlighted the {LIME_FEATURES} most influential, using {LIME_SAMPLES} perturbed "
                 f"samples.")
    return "\n\n".join(lines)

report_text = ("# Task 3 — Explainability findings (auto-generated)\n\n"
               f"Model explained: final Task 3 model "
               f"(attention = {FINAL_CFG.get('attention')}, placement = {FINAL_CFG.get('placement')}, "
               f"fine-tuning = {FINAL_CFG.get('unfreeze')}), "
               f"test accuracy = {acc:.4f}, Macro-F1 = {f1m:.4f}.\n\n"
               + describe(gc_correct, lime_correct, "Correct") + "\n\n"
               + describe(gc_wrong, lime_wrong, "Wrong / misclassified") + "\n\n"
               + "### Method notes\n\n"
               + "Grad-CAM was computed at `layer4`, the last convolutional stage. "
               + "LIME used SLIC superpixel segmentation and a local sparse linear surrogate. "
               + "The two methods are independent: Grad-CAM is gradient-based and needs the model's "
               + "internals, LIME treats the model as a black box.\n")

xai_md = os.path.join(RESULTS_DIR, "task3_explainability_findings.md")
with open(xai_md, "w") as f:
    f.write(report_text)

print(report_text)
print("\nSaved to:", xai_md)


In [ ]:
# ============================================================
# 13. Per-class error analysis — which classes does the model confuse?
# ============================================================
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Final model — confusion matrix (test set)")
plt.xticks(rotation=75); plt.yticks(rotation=0); plt.tight_layout()
plt.savefig(os.path.join(XAI_DIR, "confusion_matrix_for_error_analysis.png"), dpi=150,
            bbox_inches="tight")
plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
pairs = [(CLASSES[i], CLASSES[j], int(off[i, j]))
         for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if off[i, j] > 0]
pairs.sort(key=lambda t: -t[2])
print("Most confused class pairs (true -> predicted, count):")
for t, p, n in pairs[:8]:
    print(f"  {t:<25s} -> {p:<25s} {n}")
if pairs:
    print(f"\nFor the report: the dominant confusion is {pairs[0][0]} being read as {pairs[0][1]} "
          f"({pairs[0][2]} images). The Grad-CAM panels above show what the model was looking at "
          f"when it made that call.")
pd.DataFrame(pairs, columns=["true_class", "predicted_as", "count"]).to_csv(
    os.path.join(RESULTS_DIR, "task3_confusion_pairs.csv"), index=False)


In [ ]:
# ============================================================
# 14. Bundle and download the explainability artifacts
# ============================================================
ZIP_PATH = "/content/Group03_PRBD_task3_explainability_outputs.zip"
os.system(f"cd /content && rm -f {ZIP_PATH} && zip -qr {ZIP_PATH} work/figures work/results")
print("Bundled:", ZIP_PATH, f"({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")
print("\nExplainability figures produced:")
for f in sorted(os.listdir(XAI_DIR)):
    print("  figures/explainability/" + f)

try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception as e:
    print("\nAuto-download unavailable; grab it from the Files pane:", ZIP_PATH, e)


## Summary — what goes in the report

**Required by Task 3, produced above:**

* `figures/explainability/gradcam_correct_prediction.png` — Grad-CAM, correct prediction.
* `figures/explainability/gradcam_wrong_prediction.png` — Grad-CAM, wrong prediction, including the
  map for the **true** class alongside the predicted one.
* `figures/explainability/lime_correct_prediction.png` and `lime_wrong_prediction.png`.
* `figures/explainability/combined_gradcam_lime_correct_vs_wrong.png` — the single 2×4 panel that
  satisfies the requirement on its own; this is the one to put in the report body.
* `results/task3_explainability_findings.md` — auto-written interpretation with the real numbers
  filled in. Edit it for style; do not paste it unread.
* `results/task3_confusion_pairs.csv` — the ranked class confusions that the error analysis refers to.

**How to write it up.** For the correct prediction, state what Grad-CAM localised and whether LIME's
superpixels agreed (the IoU is printed). For the wrong prediction, the interesting question is
*which kind* of failure it is: if the Grad-CAM map for the predicted class sits in roughly the same
place as the map for the true class, the model found the right region and misread the texture — a
fine-grained discrimination failure. If the maps sit in different places, it locked onto the wrong
region entirely. Say which one your figure shows, and connect it to the dominant confusion pair
printed in section 13.

**A caveat worth including** (markers reward it): Grad-CAM at `layer4` has 7×7 spatial resolution
upsampled to 224×224, so its heat-maps are inherently coarse and should not be read as pixel-precise
evidence. LIME's explanations depend on the superpixel segmentation and are stochastic — re-running
with a different seed shifts the highlighted regions somewhat. Neither method proves causation about
the model's reasoning; they are approximations, and agreement between two independent approximations
is the strongest claim available here.

**Deliverable layout for GitHub:**

```
code/task3/Group03_PRBD_task3_improvement_ablation.ipynb
code/task3/Group03_PRBD_task3_explainability.ipynb
models/final_model_best.pth
report/task3/Group03_PRBD_task3_report.pdf
```
